# 04: LRCN + Data Augmentation (v2 — improved)
**Group 11 · COSE474 Deep Learning · Korea University · Spring 2026**

## Overview

This is the **improved version** of notebook 04. The original (v1) showed severe overfitting — train accuracy reached 35% while val stayed at 5.43%, worse than the LRCN baseline (8.14%). 

**What went wrong in v1:**
- Augmentation was too aggressive (all transforms applied simultaneously)
- Temporal augmentation (frame speed variation) disrupted the temporal signal the LSTM needs
- No label smoothing — model became overconfident on training clips
- StepLR scheduler was too abrupt for small dataset training

**Fixes applied in v2:**
1. Milder spatial augmentation — fewer transforms, smaller magnitudes
2. Removed temporal augmentation — preserves LSTM temporal signal
3. Label smoothing (0.1) — prevents overconfidence
4. Stronger dropout (0.3 instead of 0.2) — more regularization
5. Cosine annealing scheduler — gentler LR decay for small datasets
6. Uses plain KSLDataset — no temporal augmentation wrapper

**Compare against:**
- LRCN baseline (nb 03): 8.14%
- LRCN + aug v1 (nb 04 original): 5.43%

**Ablation table row this fills:**
```
LRCN + Augmentation v2 | Temporal ✓ | Aug ✓ (mild) | Transfer ✗ | Accuracy: TBD
```

**Outputs:**
```
models/checkpoints/lrcn_augmented_v2.pth
results/figures/04_augmentation_v2_curves.png
results/figures/04_augmentation_v2_confusion.png
results/logs/04_augmentation_v2_log.csv
```

# Setup — Elice AI Cloud

In [ ]:
# Author: Keira Hakeemi
# Notebook 04v2 — LRCN + Improved Data Augmentation
# Running on Elice AI Cloud

import os, sys
sys.path.append('/home/elicer')
import config

# override paths for Elice local storage
config.DATA_FRAMES  = '/home/elicer/frames'
config.MODELS_CKPT  = '/home/elicer/models/checkpoints'
config.MODELS_FINAL = '/home/elicer/models/final'
config.RESULTS_LOGS = '/home/elicer/results/logs'
config.RESULTS_FIGS = '/home/elicer/results/figures'
config.CKPT_AUG     = f'{config.MODELS_CKPT}/lrcn_augmented_v2.pth'

# create output dirs
for path in [config.MODELS_CKPT, config.MODELS_FINAL,
             config.RESULTS_LOGS, config.RESULTS_FIGS]:
    os.makedirs(path, exist_ok=True)

print(f"Frames : {config.DATA_FRAMES}")
print(f"Classes: {config.NUM_CLASSES}")
print(f"Frames/clip: {config.NUM_FRAMES}")

# verify frames exist
clips = os.listdir(config.DATA_FRAMES)
print(f"Total clips found: {len(clips)}")

# Imports

In [ ]:
import csv, random, datetime
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# Dataset

Using plain `KSLDataset` — no temporal augmentation wrapper.

**Why removed temporal augmentation:**
Frame speed variation disrupts the temporal signal the LSTM relies on.
With only 16 frames per clip, randomly adjusting the sampling window
causes frame repetition or skipping — the LSTM sees a corrupted sequence
rather than a natural sign motion. This was a key contributor to v1 failure.

In [ ]:
class KSLDataset(Dataset):
    """
    Standard dataset — identical to notebooks 01, 02, 03.
    Augmentation applied via transforms only (no temporal aug).
    """
    def __init__(self, clip_dirs, transform=None):
        self.clips     = clip_dirs
        self.transform = transform

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path   = self.clips[idx]
        folder_name = os.path.basename(clip_path)
        class_id    = int(folder_name.split('_')[1])
        label       = class_id - 1

        frame_files = sorted(os.listdir(clip_path))
        if len(frame_files) < config.NUM_FRAMES:
            frame_files += [frame_files[-1]] * (config.NUM_FRAMES - len(frame_files))
        frame_files = frame_files[:config.NUM_FRAMES]

        tensors = []
        for fname in frame_files:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)

        return torch.stack(tensors), label

# Signer-based Train/Val Split
Identical to all other notebooks — signers 00-15 train, 16-19 val.

In [ ]:
TRAIN_SIGNERS = {f"{i:02d}" for i in range(16)}
VAL_SIGNERS   = {f"{i:02d}" for i in range(16, 20)}

all_clips = sorted([
    os.path.join(config.DATA_FRAMES, d)
    for d in os.listdir(config.DATA_FRAMES)
    if os.path.isdir(os.path.join(config.DATA_FRAMES, d))
])

train_clips = [c for c in all_clips if os.path.basename(c).split('_')[0] in TRAIN_SIGNERS]
val_clips   = [c for c in all_clips if os.path.basename(c).split('_')[0] in VAL_SIGNERS]

print(f"Total clips : {len(all_clips)}")
print(f"Train clips : {len(train_clips)}  (signers 00-15)")
print(f"Val clips   : {len(val_clips)}   (signers 16-19)")

# Transforms — Improved Augmentation

**v1 problem:** all transforms applied simultaneously → too much distortion

**v2 fix:** milder transforms, no crop (could cut off hands), no temporal aug

**Changes from v1:**
- Flip probability: 0.5 → 0.3 (less frequent)
- Rotation: ±15° → ±8° (smaller — KSL signs performed upright)
- Color jitter: 0.3 → 0.15 (subtler)
- Removed RandomResizedCrop — cropping cuts off hand regions
- Removed temporal augmentation entirely

**Val transforms:** unchanged — clean resize + normalize only.

In [ ]:
# ── Train — mild augmentation ────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),          # less frequent than v1
    transforms.RandomRotation(degrees=8),             # smaller than v1 (was 15)
    transforms.ColorJitter(
        brightness=0.15,                              # subtler than v1 (was 0.3)
        contrast=0.15
    ),
    # NO RandomResizedCrop — cropping removes hand regions
    transforms.ToTensor(),
    transforms.Normalize(mean=config.NORMALIZE_MEAN,
                         std=config.NORMALIZE_STD)
])

# ── Val — clean, identical to all other notebooks ───────────
val_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.NORMALIZE_MEAN,
                         std=config.NORMALIZE_STD)
])

BATCH = 8  # same as notebooks 03/04 — memory constraint

train_dataset = KSLDataset(train_clips, train_transform)
val_dataset   = KSLDataset(val_clips,   val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
frames, lbl = train_dataset[0]
print(f"Clip shape    : {frames.shape}")  # (16, 3, 224, 224)
print()
print("Augmentation summary (v2):")
print("  RandomHorizontalFlip p=0.3  (was 0.5)")
print("  RandomRotation ±8°          (was ±15°)")
print("  ColorJitter 0.15            (was 0.3)")
print("  No crop, no temporal aug    (removed from v1)")

# Model — LRCN with stronger regularization

Same architecture as notebook 03 (Marcello) with two changes:
- Dropout increased from **0.2 → 0.3** to combat the overfitting seen in v1
- Everything else identical to maintain valid ablation comparison

In [ ]:
class LRCN(nn.Module):
    """
    VGG16 (frozen, ImageNet) -> per-frame feature -> LSTM -> classifier.
    Based on 03_CNN_LSTM.ipynb (Marcello).
    Change from v1: dropout 0.2 -> 0.3 to reduce overfitting.
    """
    def __init__(self,
                 num_classes  = config.NUM_CLASSES,
                 feature_dim  = 512,
                 lstm_hidden  = config.LSTM_HIDDEN,
                 lstm_layers  = config.LSTM_LAYERS,
                 bidirectional= config.LSTM_BIDIRECT):
        super(LRCN, self).__init__()

        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.avgpool  = vgg.avgpool

        for p in self.features.parameters():
            p.requires_grad = False

        # dropout increased to 0.3 (from 0.2 in nb 03)
        self.frame_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.lstm = nn.LSTM(
            input_size    = feature_dim,
            hidden_size   = lstm_hidden,
            num_layers    = lstm_layers,
            batch_first   = True,
            bidirectional = bidirectional,
        )

        out_dim = lstm_hidden * (2 if bidirectional else 1)
        # dropout increased to 0.3 (from 0.2 in nb 03)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(out_dim, num_classes),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        with torch.no_grad():
            x = self.features(x)
            x = self.avgpool(x)
        x = self.frame_fc(x)
        x = x.view(B, T, -1)
        out, (h_n, _) = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)


model = LRCN().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {total - trainable:,}")
print(f"Total params     : {total:,}")

# Training Setup — improved

**Changes from v1:**
- `CrossEntropyLoss(label_smoothing=0.1)` — prevents overconfidence on small dataset
- `CosineAnnealingLR` instead of `StepLR` — gentler decay, better for small datasets

**Label smoothing explained:**
Instead of training with hard labels (correct class = 1.0, others = 0.0),
label smoothing uses soft labels (correct class = 0.9, others = 0.1/76).
This prevents the model from becoming overconfident, which is a key
cause of overfitting on small datasets.

In [ ]:
# label smoothing — key fix for overfitting on small dataset
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=config.WEIGHT_DECAY
)

# cosine annealing — gentler than StepLR for small datasets
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=1e-6
)

print(f"Loss      : CrossEntropyLoss (label_smoothing=0.1)")
print(f"Optimizer : Adam (lr=1e-4, wd={config.WEIGHT_DECAY})")
print(f"Scheduler : CosineAnnealingLR (T_max={config.NUM_EPOCHS}, eta_min=1e-6)")
print(f"Dropout   : 0.3 (was 0.2 in nb 03)")
print(f"Epochs    : {config.NUM_EPOCHS}, Patience : {config.PATIENCE}")

# Train and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(clips)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += logits.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            logits = model(clips)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds       = logits.argmax(1)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100.0 * correct / total, all_preds, all_labels

## Training Loop

**Watch for:**
- Train/val gap should be narrower than v1 (v1 had train 35%, val 5%)
- Val accuracy should be >= 8.14% (LRCN baseline) to show augmentation helps
- If val plateaus around 8%, augmentation is neutral — also valid finding

In [ ]:
history = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc']}
best_val_acc, patience_count = 0.0, 0
best_preds, best_labels      = [], []

print("Starting training (LRCN + Augmentation v2)...\n")
print(f"{'Epoch':>5} {'Tr Loss':>9} {'Tr Acc':>8} {'Vl Loss':>9} {'Vl Acc':>8}")
print('─' * 48)

for epoch in range(1, config.NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc, vl_preds, vl_labels = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    for k, v in zip(['train_loss','train_acc','val_loss','val_acc'],
                    [tr_loss, tr_acc, vl_loss, vl_acc]):
        history[k].append(v)

    marker = ' <-- best' if vl_acc > best_val_acc else ''
    print(f"{epoch:>5} {tr_loss:>9.4f} {tr_acc:>7.2f}% {vl_loss:>9.4f} {vl_acc:>7.2f}%{marker}")

    if vl_acc > best_val_acc:
        best_val_acc   = vl_acc
        best_preds     = vl_preds
        best_labels    = vl_labels
        patience_count = 0
        torch.save(model.state_dict(), config.CKPT_AUG)
    else:
        patience_count += 1

    if patience_count >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nBest val accuracy  : {best_val_acc:.2f}%")
print(f"LRCN baseline (03) : 8.14%")
print(f"Aug v1 result      : 5.43%")
print(f"Delta from LRCN    : {best_val_acc - 8.14:+.2f} pp")
print(f"Delta from v1      : {best_val_acc - 5.43:+.2f} pp")

# Results
## Training curves

Compare train/val gap against v1 (train 35% vs val 5%).
A narrower gap means the fixes reduced overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('LRCN + Aug v2 — Loss'); ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.axhline(y=8.14, color='gray', linestyle='--', label='LRCN baseline 8.14%')
ax2.axhline(y=5.43, color='red',  linestyle='--', label='Aug v1 5.43%')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('LRCN + Aug v2 — Accuracy'); ax2.legend()

plt.tight_layout()
path = f"{config.RESULTS_FIGS}/04_augmentation_v2_curves.png"
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

## Confusion matrix

In [ ]:
cm = confusion_matrix(best_labels, best_preds)
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=range(1, 78), yticklabels=range(1, 78))
plt.title(f'LRCN + Aug v2 — Confusion Matrix  (Val Acc: {best_val_acc:.2f}%)', fontsize=13)
plt.ylabel('True class'); plt.xlabel('Predicted class')
plt.tight_layout()
path = f"{config.RESULTS_FIGS}/04_augmentation_v2_confusion.png"
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

# Log Results

In [ ]:
row = {
    'experiment':        '04_lrcn_augmentation_v2',
    'date':              datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model':             'LRCN + mild aug + label smoothing + cosine LR',
    'temporal_modeling': True,
    'augmentation':      True,
    'aug_version':       'v2_mild',
    'transfer_learning': False,
    'aug_flip':          0.3,
    'aug_rotation':      8,
    'aug_brightness':    0.15,
    'temporal_aug':      False,
    'label_smoothing':   0.1,
    'dropout':           0.3,
    'scheduler':         'CosineAnnealingLR',
    'train_signers':     '00-15',
    'val_signers':       '16-19',
    'lstm_hidden':       config.LSTM_HIDDEN,
    'batch_size':        BATCH,
    'learning_rate':     1e-4,
    'best_val_acc':      round(best_val_acc, 2),
    'lrcn_baseline_acc': 8.14,
    'aug_v1_acc':        5.43,
    'delta_from_lrcn':   round(best_val_acc - 8.14, 2),
    'delta_from_v1':     round(best_val_acc - 5.43, 2),
    'epochs_run':        len(history['val_acc']),
    'notes':             'Mild aug + label smoothing + cosine LR + dropout 0.3. No temporal aug.'
}

log_path = f"{config.RESULTS_LOGS}/04_augmentation_v2_log.csv"
write_header = not os.path.exists(log_path)
with open(log_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header: writer.writeheader()
    writer.writerow(row)

print("Results logged:")
for k, v in row.items(): print(f"  {k}: {v}")
print(f"\nSaved to: {log_path}")

# Summary

In [ ]:
print('=' * 58)
print('  LRCN + AUGMENTATION v2 COMPLETE')
print('=' * 58)
print(f"  Best val accuracy  : {best_val_acc:.2f}%")
print(f"  LRCN baseline (03) : 8.14%")
print(f"  Aug v1 result      : 5.43%  (aggressive aug, overfit)")
print(f"  Delta from LRCN    : {best_val_acc - 8.14:+.2f} pp")
print(f"  Delta from v1      : {best_val_acc - 5.43:+.2f} pp")
print()
print("  Ablation table so far:")
print(f"  Simple CNN          :  6.98%  (notebook 02)")
print(f"  LRCN baseline       :  8.14%  (notebook 03)")
print(f"  LRCN + aug v1       :  5.43%  (04 original — too aggressive)")
print(f"  LRCN + aug v2 (this):  {best_val_acc:.2f}%  (04 improved)")
print(f"  LRCN + transfer     :  TBD    (notebook 05 — Marcello)")
print(f"  LRCN + best combo   :  TBD    (notebook 06 — Keira)")
print()
print(f"  Checkpoint : {config.CKPT_AUG}")
print(f"  Curves     : results/figures/04_augmentation_v2_curves.png")
print(f"  Confusion  : results/figures/04_augmentation_v2_confusion.png")
print(f"  Log        : results/logs/04_augmentation_v2_log.csv")
print()
print("  --> Next: 05_transfer_learning.ipynb (Marcello)")
print('=' * 58)